### Setup

In [1]:
import matplotlib.pyplot as plt
import scipy as sc
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
import seaborn as sns
import sys
import os
import torch
from tqdm import tqdm
import scipy.optimize as opt

In [2]:
sys.path.insert(0, os.getcwd())

from model import GPTConfig, GPT
from lib.utils import get_batch

In [3]:
device='xpu'
dataset='shakespeare_char'
checkpoint_dir='out-shakespeare-char'

In [4]:
%%capture
torch.manual_seed(1337)

# load the checkpointed model state from last train save
ckpt_path = os.path.join(checkpoint_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.eval() # disables dropout
model.to(device)

In [5]:
model.config

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=4, head_size=20, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)

In [6]:
n = model.config.block_size
h = model.config.n_head
hs = model.config.head_size

In [7]:
X, Y = get_batch('eval', os.path.join('data', dataset), device, n, model.config.batch_size)
A,T,v = model.get_matricies(X,0)

In [8]:
T.shape

(32, 20)

In [9]:
np.linalg.matrix_rank(T)

np.int64(20)

In [10]:
T_aug = np.concatenate([T, np.ones((n, 1))], axis=1)
basis_LN_base = sc.linalg.null_space(T.T)
basis_LN = sc.linalg.null_space(T_aug.T)

In [11]:
basis_LN_base.shape

(32, 12)

In [12]:
d = basis_LN[:,0]
a = A[:,1]

In [13]:
def get_lam_constraints(d):
    lam_max_list = []
    lam_min_list = []
    for idx in range(len(d)):
        if d[idx] < 0:
            lam_max_list.append(-a[idx] / d[idx])
        elif d[idx] > 0:
            lam_min_list.append(-a[idx] / d[idx])
    
    lam_max = np.min(lam_max_list)
    lam_min = np.max(lam_min_list)
    return lam_min, lam_max

In [14]:
# min max constraints for lambda
rows= []
for i in range(basis_LN.shape[1]):
    d = basis_LN[:,i]
    lam_min, lam_max = get_lam_constraints(d)
    rows.append({'min':lam_min, 'max':lam_max})
df = pd.DataFrame(rows)
df

,min,max
0,-0.000000,0.130847
1,-0.000000,0.114001
2,-0.000000,0.100485
3,-0.000000,0.085135
4,-0.055867,0.000000
5,-0.000000,0.083385
6,-0.000000,0.092476
7,-0.000000,0.128857
8,-0.000000,0.114129
9,-0.000000,0.121308


In [15]:
def objective(x : np.array):
    return np.max(np.abs(x - x[0]))

In [16]:
# Minimize the objective function via enumeration method
a = A[:,1]
for i in range(basis_LN.shape[1]):
    d = basis_LN[:,i]
    lam_min, lam_max = get_lam_constraints(d)

    obj_min = 1e12
    opt_lam = None
    for lam in [lam_min, lam_max]:
        vals = np.zeros(basis_LN_base.shape[1])
        for j in range(basis_LN_base.shape[1]):
            vals[j] = (basis_LN_base[:,j].T  @ (a + (lam * d))) / (basis_LN_base[:,j].T @ np.ones(n))

        obj = objective(vals)
        if(obj < obj_min):
            obj_min = obj
            opt_lam = lam

    vals = np.array([
        (basis_LN_base[:,j].T @ (a + (opt_lam * d))) / (basis_LN_base[:,j].T @ np.ones(n))
        for j in range(basis_LN_base.shape[1])
    ])
    print(obj_min)
    a = a + opt_lam * d

2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902
2.390394482208902


In [17]:
a_orig = A[:, 1].copy()
B = basis_LN        # feasible directions: shape (n, k)
E = basis_LN_base   # measurement directions: shape (n, m)

k = B.shape[1]
m = E.shape[1]

# Precompute LN-invariant measurement matrix
# vals[j] = (E[:,j] @ x) / (E[:,j] @ ones)
ones = np.ones(n)
e_dot_1 = E.T @ ones          # shape (m,)
# vals[j](lambda) = (E[:,j] @ a + E[:,j] @ B @ lambda) / e_dot_1[j]
E_B = (E.T @ B) / e_dot_1[:, None]   # shape (m, k)
e_a = (E.T @ a_orig) / e_dot_1        # shape (m,)

# Objective: minimize t = max_j |vals[j] - vals[0]|
# Variables: [lambda (k), t (1)]
c = np.zeros(k + 1)
c[-1] = 1.0

# vals[j] - vals[0] <= t  and  -(vals[j] - vals[0]) <= t  for j=1..m-1
diffs_E  = E_B[1:, :] - E_B[0, :]   # shape (m-1, k)
a_diffs  = e_a[1:] - e_a[0]          # shape (m-1,)
ones_t   = -np.ones((m - 1, 1))

A_ub = np.block([
    [ diffs_E, ones_t],
    [-diffs_E, ones_t]
])
b_ub = np.concatenate([-a_diffs, a_diffs])

# Non-negativity of attention weights: a + B @ lambda >= 0
A_nonneg = -B                         # shape (n, k)
b_nonneg = a_orig                     # shape (n,)
A_ub_full = np.block([
    [A_ub],
    [np.hstack([A_nonneg, np.zeros((n, 1))])]
])
b_ub_full = np.concatenate([b_ub, b_nonneg])

result = opt.linprog(c, A_ub=A_ub_full, b_ub=b_ub_full,
                     bounds=[(None, None)] * k + [(0, None)])

lam_opt = result.x[:k]
a_opt   = a_orig + B @ lam_opt
print("Optimal objective:", result.fun)

Optimal objective: 0.0


In [22]:
a_opt

array([0.00084913, 0.4947235 , 0.33343615, 0.24481453, 0.20085065,
       0.17356369, 0.14208029, 0.12576963, 0.13247537, 0.05643378,
       0.06480539, 0.10868704, 0.09996835, 0.04735027, 0.08184994,
       0.03137522, 0.06762577, 0.03639371, 0.0402335 , 0.05381893,
       0.07221512, 0.02177754, 0.04775427, 0.04639968, 0.06077468,
       0.02537003, 0.03515645, 0.0166213 , 0.03025701, 0.05016807,
       0.02139446, 0.02440353])

In [25]:
lam_opt

array([ 3.43968375e-03,  8.49202445e-04, -7.61354461e-04,  4.54014528e-05,
       -3.28783529e-03,  1.70535596e-03, -1.73719748e-03,  6.33715056e-03,
        3.14892939e-03,  6.14482082e-03, -4.74416863e-03])

In [20]:
a_orig = A[:, 1].copy()
B = basis_LN        # shape (n, k)
E = basis_LN_base   # shape (n, m)

k = B.shape[1]
m = E.shape[1]
n = len(a_orig)

e_dot_1 = E.T @ np.ones(n)  # shape (m,)

def get_x(lam):
    return a_orig + B @ lam

def phi(x):
    """Safe log-based LN-invariant coordinates."""
    if np.any(x <= 0):
        return None
    return (E.T @ np.log(x)) / e_dot_1  # shape (m,)

def objective(lam):
    x = get_x(lam)
    vals = phi(x)
    if vals is None:
        return 1e12
    return np.max(np.abs(vals - vals[0])) 

In [21]:
a_orig = A[:, 1].copy()
B = basis_LN
E = basis_LN_base

k = B.shape[1]
n = len(a_orig)
e_dot_1 = E.T @ np.ones(n)
eps = 1e-8

def get_x(lam):
    return a_orig + B @ lam

def objective(lam):
    x = get_x(lam)
    if np.any(x <= 0):
        return 1e12
    vals = (E.T @ np.log(x)) / e_dot_1
    return float(np.max(np.abs(vals - vals[0])))

# Step 1: LP warm start — find interior feasible point
# maximize s s.t. a + B @ lam >= s, i.e. -B @ lam + s <= a
c_feas  = np.zeros(k + 1); c_feas[-1] = -1.0
A_feas  = np.hstack([-B, np.ones((n, 1))])
lp      = opt.linprog(c_feas, A_ub=A_feas, b_ub=a_orig,
                  bounds=[(None,None)]*k + [(None,None)],
                  method='highs')
lam0    = lp.x[:k]
print(f"LP slack:       {-lp.fun:.6f}")
print(f"Objective(lam0): {objective(lam0):.6f}")

# Step 2: SLSQP with multiple restarts
results = []
for trial in range(50):
    lam_init = lam0 + np.random.randn(k) * (0 if trial == 0 else 0.01)
    if np.any(get_x(lam_init) <= eps):
        continue
    res = opt.minimize(
        objective,
        lam_init,
        method='SLSQP',
        constraints={'type': 'ineq', 'fun': lambda l: get_x(l) - eps},
        options={'ftol': 1e-12, 'maxiter': 5000, 'disp': False}
    )
    if res.fun < 1e11:  # filter out infeasible
        results.append(res)

best = min(results, key=lambda r: r.fun)
a_opt = get_x(best.x)

print(f"\nOptimal objective: {best.fun:.6f}")
print(f"Min component:     {np.min(a_opt):.8f}")
print(f"Solver success:    {best.success}")
print(f"Solver message:    {best.message}")

LP slack:       0.002620
Objective(lam0): 1934.710498

Optimal objective: 48.567901
Min component:     0.00084913
Solver success:    True
Solver message:    Optimization terminated successfully


Optimal objective: 0.000229
Min component:     0.00037876
Solver success:    True
Solver message:    Optimization terminated successfully
Training:  70%|███████████████████████████████████████████████████████████████████▉                             | 3500/5000 [01:08<00:16, 88.39it/s, loss=2.1738]

Best Result from this:

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=4, head_size=20, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)
